# Extraction Test Notebook
Validate point time series extraction from the preprocessed Landsat collection.

**Steps**
1. Init GEE & build collection
2. Extract single point — clear-sky LST only
3. Extract single point — LST + cloud flag
4. Extract multiple points
5. Quick visual check
6. Save results

## 1. Setup

In [1]:
import ee
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Anchor all paths to the project root (one level up from notebooks/)
ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent

from rs_timeseries.config import load_config, get_ee_project
from rs_timeseries.io import build_aoi, build_merged_collection
from rs_timeseries.preprocessing import preprocess_collection
from rs_timeseries.extraction import (
    extract_point_timeseries,
    extract_multipoint_timeseries,
)

# Load config and init GEE
config = load_config(ROOT / "configs" / "swiss_alps_lst.yaml")

project_id = get_ee_project()
ee.Initialize(project=project_id) if project_id else ee.Initialize()
print(f"GEE initialized. ROOT: {ROOT}")

GEE initialized. ROOT: d:\Projects\landsat-timeseries


## 2. Build & Preprocess Collection

In [2]:
aoi        = build_aoi(config)
collection = build_merged_collection(config, aoi)
collection = preprocess_collection(collection, config)

# Verify cloud band is present
bands = collection.first().bandNames().getInfo()
print(f'Bands: {bands}')
# Expected: ['SurfT', 'Emis', 'cloud']

Bands: ['SurfT', 'Emis', 'cloud']


## 3. Single Point — Clear-Sky LST Only

In [3]:
# Test with a short date range first — expand to full range once confirmed working
LON, LAT = 8.55, 47.37   # Zurich

df_lst = extract_point_timeseries(
    collection  = collection,
    lon         = LON,
    lat         = LAT,
    bands       = ['SurfT'],
    scale       = 30,
    start_year  = 2018,
    end_year    = 2022,
    verbose     = True,
)

df_lst.head(10)

  2018: 498 scenes
  2019: 496 scenes
  2020: 258 scenes
  2021: 249 scenes
  2022: 256 scenes

Extracted 1757 scenes  (2018–2022)  | clear: 0  cloudy: 1757


,SurfT,date
0,None,2018-01-01 10:22:50.442
1,None,2018-01-01 10:23:14.333
2,None,2018-01-01 10:23:38.212
3,None,2018-01-02 10:18:39.480
4,None,2018-01-02 10:19:03.362
5,None,2018-01-02 10:19:27.244
6,None,2018-01-03 10:11:15.867
7,None,2018-01-04 10:07:03.981
8,None,2018-01-05 09:58:53.460
9,None,2018-01-08 10:28:58.507


In [8]:
print(df_lst.to_string())

     SurfT                    date
0     None 2018-01-01 10:22:50.442
1     None 2018-01-01 10:23:14.333
2     None 2018-01-01 10:23:38.212
3     None 2018-01-02 10:18:39.480
4     None 2018-01-02 10:19:03.362
5     None 2018-01-02 10:19:27.244
6     None 2018-01-03 10:11:15.867
7     None 2018-01-04 10:07:03.981
8     None 2018-01-05 09:58:53.460
9     None 2018-01-08 10:28:58.507
10    None 2018-01-08 10:29:22.394
11    None 2018-01-09 10:25:32.177
12    None 2018-01-10 10:16:35.985
13    None 2018-01-10 10:16:59.871
14    None 2018-01-11 10:12:21.050
15    None 2018-01-11 10:12:44.932
16    None 2018-01-11 10:13:08.816
17    None 2018-01-12 10:04:37.316
18    None 2018-01-12 10:05:01.194
19    None 2018-01-13 10:00:45.418
20    None 2018-01-16 10:31:12.898
21    None 2018-01-17 10:23:07.238
22    None 2018-01-17 10:23:31.125
23    None 2018-01-18 10:19:13.297
24    None 2018-01-19 10:10:20.634
25    None 2018-01-19 10:10:44.513
26    None 2018-01-19 10:11:08.395
27    None 2018-01-2

## 4. Single Point — LST + Cloud Flag

In [17]:
df_full = extract_point_timeseries(
    collection  = collection,
    lon         = LON,
    lat         = LAT,
    bands       = ['SurfT', 'cloud'],
    scale       = 30,
    start_year  = 2018,
    end_year    = 2022,
    verbose     = True,
)

print(f"\nTotal scenes : {len(df_full)}")
print(f"Clear pixels : {(df_full['cloud'] == 0).sum()}")
print(f"Cloudy pixels: {(df_full['cloud'] == 1).sum()}")
print(f"Cloud cover  : {df_full['cloud'].mean():.1%}")
df_full.head(10)

KeyboardInterrupt: 

In [16]:
print(df_full.to_string())

     SurfT cloud                    date
0     None  None 2018-01-01 10:22:50.442
1     None  None 2018-01-01 10:23:14.333
2     None  None 2018-01-01 10:23:38.212
3     None  None 2018-01-02 10:18:39.480
4     None  None 2018-01-02 10:19:03.362
5     None  None 2018-01-02 10:19:27.244
6     None  None 2018-01-03 10:11:15.867
7     None  None 2018-01-04 10:07:03.981
8     None  None 2018-01-05 09:58:53.460
9     None  None 2018-01-08 10:28:58.507
10    None  None 2018-01-08 10:29:22.394
11    None  None 2018-01-09 10:25:32.177
12    None  None 2018-01-10 10:16:35.985
13    None  None 2018-01-10 10:16:59.871
14    None  None 2018-01-11 10:12:21.050
15    None  None 2018-01-11 10:12:44.932
16    None  None 2018-01-11 10:13:08.816
17    None  None 2018-01-12 10:04:37.316
18    None  None 2018-01-12 10:05:01.194
19    None  None 2018-01-13 10:00:45.418
20    None  None 2018-01-16 10:31:12.898
21    None  None 2018-01-17 10:23:07.238
22    None  None 2018-01-17 10:23:31.125
23    None  None

In [12]:
# --- Debug: inspect raw QA values at the test point ---
import numpy as np

point = ee.Geometry.Point([8.55, 47.37])

# Take 5 images from one year
#sample = collection.filterDate("2021-01-01", "2022-01-01").limit(5)
sample = collection.filterDate("2021-07-01", "2021-09-01").limit(5)


def inspect(img):
    vals = img.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
        bestEffort=True,
    )
    return ee.Feature(None, vals).set("date", img.date().format("YYYY-MM-dd"))

fc = ee.FeatureCollection(sample.map(inspect))
rows = fc.getInfo()["features"]

for r in rows:
    p = r["properties"]
    print(f"  {p.get('date')}  SurfT={p.get('SurfT')}  cloud={p.get('cloud')}")

  2021-07-06  SurfT=None  cloud=1
  2021-07-06  SurfT=None  cloud=None
  2021-07-06  SurfT=None  cloud=None
  2021-07-08  SurfT=None  cloud=None
  2021-07-11  SurfT=None  cloud=None


## 5. Multiple Points

In [ ]:
points = [
    {'point_id': 'ZRH', 'lon': 8.55,  'lat': 47.37},  # Zurich
    {'point_id': 'GST', 'lon': 9.85,  'lat': 46.50},  # Graubünden
    {'point_id': 'WEN', 'lon': 7.92,  'lat': 46.58},  # Wengen
]

df_multi = extract_multipoint_timeseries(
    collection  = collection,
    points      = points,
    bands       = ['SurfT', 'cloud'],
    scale       = 30,
    start_year  = 2018,
    end_year    = 2022,
    save_path = str(ROOT / "data" / "timeseries" / "timeseries_test.csv"),
)

df_multi.groupby('point_id').agg(
    scenes      = ('SurfT', 'count'),
    cloud_cover = ('cloud', 'mean'),
    lst_min     = ('SurfT', 'min'),
    lst_max     = ('SurfT', 'max'),
).round(3)

## 6. Visual Check

In [8]:
fig, axes = plt.subplots(len(points), 1, figsize=(14, 3.5 * len(points)), sharex=True)

colors = {'ZRH': 'steelblue', 'GST': 'darkorange', 'WEN': 'seagreen'}

for ax, pt in zip(axes, points):
    pid = pt['point_id']
    sub = df_multi[df_multi['point_id'] == pid].copy()

    clear  = sub[sub['cloud'] == 0]
    cloudy = sub[sub['cloud'] == 1]

    ax.scatter(clear['date'],  clear['SurfT'],  s=8, alpha=0.7,
               color=colors[pid], label='Clear')
    ax.scatter(cloudy['date'], cloudy['SurfT'], s=8, alpha=0.3,
               color='gray', label='Cloudy (masked)')

    ax.set_ylabel('LST (K)')
    ax.set_title(f"{pid}  [{pt['lon']}, {pt['lat']}]")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.legend(markerscale=2, frameon=False)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "data" / "timeseries" / "extraction_test.png", dpi=150, bbox_inches="tight")
plt.show()
print('Plot saved.')

NameError: name 'points' is not defined

## 7. Full Time Series (1984–2024)
Run this cell only after the test above confirms extraction is working correctly.

In [9]:
df_full_range = extract_multipoint_timeseries(
    collection  = collection,
    points      = points,
    bands       = ['SurfT', 'cloud'],
    scale       = 30,
    start_year  = 1984,
    end_year    = 2024,
    save_path = str(ROOT / "data" / "timeseries" / "timeseries_all.csv"),
)

print(df_full_range.groupby('point_id')[['SurfT', 'cloud']].describe().round(2))

NameError: name 'points' is not defined